In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','xgboost==3.1.3','pandas==2.2.3','numpy==2.3.5','pyarrow==26.0.0','holidays==0.106'])


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/followup_run.py')
p.write_text('"""CUDA combinations, key factorial and new historical baselines. No new data upload."""\nimport ast,gc,hashlib,json,subprocess,time,types,warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport holidays\nimport xgboost as xgb\n\nMODELS=[\'catboost_log_level\',\'lightgbm_log_diff\',\'lightgbm_log_level\',\'xgboost_log_diff\']\nOUT=Path(\'/kaggle/working\')\n\ndef main():\n    derived=list(Path(\'/kaggle/input\').rglob(\'research_manifest.json\'))\n    originals=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))\n    assert len(derived)==len(originals)==1 and xgb.__version__==\'3.1.3\'\n    data=derived[0].parent;source=originals[0].parent\n    plan=json.loads(derived[0].read_text());original=json.loads(originals[0].read_text())\n    for name,digest in original[\'input_hashes\'].items():\n        assert hashlib.sha256((source/name).read_bytes()).hexdigest()==digest\n    for name,digest in plan[\'hashes\'].items():\n        assert hashlib.sha256((data/name).read_bytes()).hexdigest()==digest\n    notebook=json.loads((source/\'original_solution.ipynb\').read_text())\n    namespace=dict(holidays=holidays)\n    for i in [10,14,16,20,26,37]:\n        exec(compile(\'\'.join(notebook[\'cells\'][i][\'source\']),\'original:\'+str(i),\'exec\'),namespace)\n    for node in ast.parse(\'\'.join(notebook[\'cells\'][18][\'source\'])).body:\n        if isinstance(node,ast.FunctionDef):\n            exec(compile(ast.Module(body=[node],type_ignores=[]),\'original_setup\',\'exec\'),namespace)\n        elif isinstance(node,ast.Assign) and isinstance(node.targets[0],ast.Name):\n            if node.targets[0].id in {\'COMMON_BASE_FEATURE_KWARGS\',\'ID_COL\',\'YEAR_COL\',\'MONTH_COL\',\'TIME_COL\',\'TARGET_COL\'}:\n                namespace[node.targets[0].id]=ast.literal_eval(node.value)\n    rename=ast.literal_eval(ast.parse(\'\'.join(notebook[\'cells\'][8][\'source\'])).body[0].value)\n    raw=pd.read_csv(source/\'train_2.csv\').rename(columns=rename)\n    assert len(raw)==485082\n    base=plan[\'base_columns\'];params=plan[\'parameters\'];records=[]\n    extra_columns=list(dict.fromkeys(c for group in plan[\'groups\'].values() for c in group))\n    all_extras=pd.concat([pd.read_parquet(data/\'2025-02_train.parquet\',columns=[\'new_id\',\'year_month_index\']+extra_columns),\n        pd.read_parquet(data/\'2025-02_holdout.parquet\',columns=[\'new_id\',\'year_month_index\']+extra_columns)])\n    assert not all_extras.duplicated([\'new_id\',\'year_month_index\']).any()\n    def add_extra(meta,columns):\n        joined=meta[[\'new_id\',\'year_month_index\']].merge(all_extras[[\'new_id\',\'year_month_index\']+columns],\n            on=[\'new_id\',\'year_month_index\'],how=\'left\',sort=False,validate=\'one_to_one\')\n        assert len(joined)==len(meta)\n        return joined[columns].set_axis(meta.index)\n    def fit(fold,config,xtrain,xvalid,ytrain,yvalid,meta,columns,baseline_mape=None):\n        selected=base+columns;train_mask=np.isfinite(ytrain);valid_mask=np.isfinite(yvalid)\n        start=time.perf_counter();model=xgb.XGBRegressor(**params)\n        model.fit(xtrain.loc[train_mask,selected],np.asarray(ytrain)[train_mask],\n            eval_set=[(xvalid.loc[valid_mask,selected],np.asarray(yvalid)[valid_mask])],verbose=False)\n        device=json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n        assert device.startswith(\'cuda\')\n        prediction=meta.pto_lag_1.to_numpy()*np.exp(np.asarray(model.predict(xvalid[selected]),dtype=float))\n        assert len(prediction)==18657 and np.isfinite(prediction).all()\n        error=prediction-meta.pto.to_numpy()\n        mape=float((np.abs(error)/meta.pto.to_numpy()).mean()*100)\n        row=dict(model=\'xgboost_log_diff\',configuration=config,fold=fold,mape=mape,\n            mae=float(np.abs(error).mean()),rmse=float(np.sqrt((error**2).mean())),\n            bias_pct=float((error/meta.pto.to_numpy()).mean()*100),best_iteration=int(model.best_iteration),\n            seconds=time.perf_counter()-start,backend=\'CUDA\',device=device,xgboost_version=xgb.__version__,\n            parameters=params,features_added=columns,notebook_sha256=plan[\'notebook_sha256\'],\n            validation_only=True,leakage_check=\'passed\',baseline_mape=baseline_mape,\n            source_feature_checks=plan[\'feature_checks\'],phase=\'followup\')\n        directory=OUT/\'results\'/fold/\'xgboost_log_diff\';directory.mkdir(parents=True,exist_ok=True)\n        table=meta[[\'new_id\',\'year_month_index\',\'region\',\'settlement\',\'pto\']].copy();table[\'prediction\']=prediction\n        table.to_csv(directory/(config+\'.csv\'),index=False)\n        (directory/(config+\'.json\')).write_text(json.dumps(row,indent=2))\n        model.save_model(directory/(config+\'.ubj\'));records.append(row)\n        (OUT/\'phase2_metrics.json\').write_text(json.dumps(records,indent=2))\n        print(\'RESULT\',fold,config,mape,flush=True);del model;gc.collect()\n    def build(fold,key=False,frame=None):\n        year,month=map(int,fold.split(\'-\'));r=raw.copy() if frame is None else frame.copy()\n        if key:r[\'settlement\']=r.region.astype(str)+\' :: \'+r.settlement.astype(str)\n        with warnings.catch_warnings():\n            warnings.simplefilter(\'ignore\',pd.errors.PerformanceWarning)\n            full=namespace[\'build_retail_features_with_split\'](r,split_mode=\'train_val\',min_year=2023,\n                val_year=year,val_month=month,base_feature_kwargs=namespace[\'COMMON_BASE_FEATURE_KWARGS\'],verbose=False)\n        return full\n    def compact(full):\n        requested={\'new_id\',\'year_month_index\',\'pto\',\'pto_lag_1\',\'region\',\'settlement\'}\n        for model in MODELS:\n            for name in namespace[model+\'_features\']:\n                requested.add(name[:-3] if name.endswith(\'_cb\') else name)\n                if name.endswith(\'_cat_cb\'):requested.add(name[:-7])\n        result={key:None for key in [\'train_features\',\'val_features\',\'test_train_features\',\'test_features\']}\n        for key in [\'train_features\',\'val_features\']:\n            result[key]=full[key][[c for c in full[key] if c in requested]].copy()\n            for name in namespace[\'catboost_log_level_features\']:\n                if name not in result[key] and (\'feb_mar_growth_2024\' in name or \'region_march_growth\' in name):result[key][name]=np.nan\n        result.update(categorical_cols=full[\'categorical_cols\'],feature_cols=full[\'feature_cols\'])\n        return result\n    # Source-key leakage check, including changed current and all later outcomes.\n    ids=np.random.default_rng(2026).choice(raw.new_id.unique(),160,replace=False)\n    sample=raw[raw.new_id.isin(ids)]\n    for fold in plan[\'folds\']:\n        year,month=map(int,fold.split(\'-\'));changed=sample.copy()\n        changed.loc[changed.year*12+changed.month>=year*12+month,\'pto\']*=23\n        a=compact(build(fold,True,sample));b=compact(build(fold,True,changed))\n        pa=namespace[\'_prepare_numeric_model_pack\'](a,selected_features=base,target_mode=\'growth_log\',label=\'xgboost_log_diff\',verbose=False)[\'validation\']\n        pb=namespace[\'_prepare_numeric_model_pack\'](b,selected_features=base,target_mode=\'growth_log\',label=\'xgboost_log_diff\',verbose=False)[\'validation\']\n        for part in [\'X_train\',\'X_val\']:\n            pd.testing.assert_frame_equal(pa[part][base],pb[part][base])\n    (OUT/\'phase2_checks.json\').write_text(json.dumps(dict(status=\'passed\',key_cutoffs=plan[\'folds\'],\n        extra_features=plan[\'feature_checks\'],raw_hash=original[\'input_hashes\'][\'train_2.csv\']),indent=2))\n    for fold in plan[\'folds\']:\n        train=pd.read_parquet(data/(fold+\'_train.parquet\'));valid=pd.read_parquet(data/(fold+\'_holdout.parquet\'))\n        for config in [\'knn8_history+wx_relative\',\'hybrid8_history+within_market\']:\n            columns=list(dict.fromkeys(c for g in config.split(\'+\') for c in plan[\'groups\'][g]))\n            fit(fold,config,train,valid,train.target,valid.target,valid,columns)\n        del train,valid;gc.collect()\n        full=build(fold,True);c=compact(full);del full;gc.collect()\n        directory=OUT/\'prepared\'/(\'key_\'+fold);directory.mkdir(parents=True,exist_ok=True)\n        for key,name in [(\'train_features\',\'train\'),(\'val_features\',\'holdout\')]:c[key].to_parquet(directory/(name+\'.parquet\'),index=False)\n        (directory/\'manifest.json\').write_text(json.dumps(dict(raw_sha256=original[\'input_hashes\'][\'train_2.csv\'],\n            notebook_sha256=plan[\'notebook_sha256\'],categorical_cols=c[\'categorical_cols\'],feature_cols=c[\'feature_cols\'],\n            key=\'region :: original settlement\',prepared_backend=\'Kaggle\',source_untouched=True),indent=2))\n        pack=namespace[\'_prepare_numeric_model_pack\'](c,selected_features=base,target_mode=\'growth_log\',label=\'xgboost_log_diff\',verbose=False)[\'validation\']\n        for config,groups in [(\'key\',[]),(\'key+hybrid8_history\',[\'hybrid8_history\'])]:\n            columns=[col for group in groups for col in plan[\'groups\'][group]]\n            xtrain=pd.concat([pack[\'X_train\'][base],add_extra(c[\'train_features\'],columns)],axis=1)\n            xvalid=pd.concat([pack[\'X_val\'][base],add_extra(c[\'val_features\'],columns)],axis=1)\n            fit(fold,config,xtrain,xvalid,pack[\'y_train\'],pack[\'y_val\'],c[\'val_features\'],columns)\n        del c,pack,xtrain,xvalid;gc.collect()\n    for fold in [\'2024-02\',\'2024-12\',\'2024-10\']:\n        full=build(fold);c=compact(full);del full;gc.collect()\n        directory=OUT/\'prepared\'/fold;directory.mkdir(parents=True,exist_ok=True)\n        for key,name in [(\'train_features\',\'train\'),(\'val_features\',\'holdout\')]:c[key].to_parquet(directory/(name+\'.parquet\'),index=False)\n        (directory/\'manifest.json\').write_text(json.dumps(dict(raw_sha256=original[\'input_hashes\'][\'train_2.csv\'],\n            notebook_sha256=plan[\'notebook_sha256\'],categorical_cols=c[\'categorical_cols\'],feature_cols=c[\'feature_cols\'],\n            prepared_backend=\'Kaggle\'),indent=2))\n        pack=namespace[\'_prepare_numeric_model_pack\'](c,selected_features=base,target_mode=\'growth_log\',label=\'xgboost_log_diff\',verbose=False)[\'validation\']\n        for config,groups in [(\'baseline\',[])]+([(\'hybrid8_history\',[\'hybrid8_history\']),(\'knn8_history+wx_relative\',[\'knn8_history\',\'wx_relative\'])] if fold==\'2024-10\' else []):\n            columns=list(dict.fromkeys(col for group in groups for col in plan[\'groups\'][group]))\n            xtrain=pd.concat([pack[\'X_train\'][base],add_extra(c[\'train_features\'],columns)],axis=1)\n            xvalid=pd.concat([pack[\'X_val\'][base],add_extra(c[\'val_features\'],columns)],axis=1)\n            fit(fold,config,xtrain,xvalid,pack[\'y_train\'],pack[\'y_val\'],c[\'val_features\'],columns)\n        del c,pack,xtrain,xvalid;gc.collect()\n    print(\'FOLLOWUP_CUDA_COMPLETE\',len(records),flush=True)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
subprocess.check_call([sys.executable,'-u',str(p)])
